In [1]:
# analyze_multilevel_metrics.py
"""
Analyze keyword overlaps, source distributions, depth levels,
and TF-IDF/similarity statistics for multi-level citation dataset.
Generates visualizations:
1. Keyword frequency (Set size)
2. Keyword intersections (UpSet)
3. Source distribution
4. Depth distribution
5. TF-IDF & Similarity statistics per combination
"""

import os
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from collections import Counter
import numpy as np
from statistics import mode
import re, ast

# ---------- CONFIG ----------
FILENAME = "results_multilevel_ranked/all_multilevel_ranked_combined.xlsx"
SAVE_DIR = "analysis_multilevel_outputs"
os.makedirs(SAVE_DIR, exist_ok=True)
plt.style.use("seaborn-v0_8-whitegrid")

# ---------- Helpers ----------
def extract_keywords(value):
    """Parse keywords from sub_category column."""
    if pd.isna(value):
        return []
    s = str(value)
    if s.startswith("[") and s.endswith("]"):
        try:
            return [x.strip().lower() for x in ast.literal_eval(s)]
        except Exception:
            pass
    return [x.strip().lower() for x in re.split(r"[,\n;\|/\t]+", s) if x.strip()]

# ---------- Load Data ----------
print(f"📘 Loading data from {FILENAME} ...")
df = pd.read_excel(FILENAME)
if df.empty:
    print("⚠️ No data found in the file.")
    exit()

# Add keyword list column
df["keywords"] = df["sub_category"].apply(extract_keywords)

# ---------- 1️⃣ Keyword Frequency (Set Size) ----------
all_keywords = [kw for lst in df["keywords"] for kw in lst]
set_counts = Counter(all_keywords)
set_df = pd.DataFrame(set_counts.items(), columns=["keyword", "count"]).sort_values("count", ascending=False)
set_df.to_excel(f"{SAVE_DIR}/set_size.xlsx", index=False)

plt.figure(figsize=(10, 6))
sns.barplot(y="keyword", x="count", data=set_df, palette="Blues_r")
plt.title("Keyword Frequency (Set Size)")
plt.xlabel("Count of Papers")
plt.ylabel("Keyword")
plt.tight_layout()
plt.savefig(f"{SAVE_DIR}/set_size_barplot.png", dpi=300)
plt.close()

# ---------- 2️⃣ Keyword Intersection (UpSet Plot) ----------
try:
    from upsetplot import UpSet, from_contents

    sets_dict = {k: set(df.index[df["keywords"].apply(lambda x: k in x)]) for k in set_df["keyword"]}
    data_upset = from_contents(sets_dict)

    plt.figure(figsize=(10, 6))
    UpSet(data_upset, show_counts=True).plot()
    plt.suptitle("Intersection Size: Keyword Overlaps (UpSet Plot)")
    plt.tight_layout()
    plt.savefig(f"{SAVE_DIR}/intersection_upset.png", dpi=300)
    plt.close()
except Exception as e:
    print(f"⚠️ Skipping UpSet plot (error: {e})")

# ---------- 3️⃣ Source Distribution ----------
if "source" in df.columns:
    source_summary = df.groupby(["sub_category", "source"]).size().reset_index(name="count")
    plt.figure(figsize=(12, 6))
    sns.barplot(x="sub_category", y="count", hue="source", data=source_summary)
    plt.title("Source Distribution per Keyword Combination")
    plt.xticks(rotation=45, ha="right")
    plt.xlabel("Keyword Combination")
    plt.ylabel("Number of Papers")
    plt.tight_layout()
    plt.savefig(f"{SAVE_DIR}/source_distribution.png", dpi=300)
    plt.close()

# ---------- 4️⃣ Depth Distribution ----------
if "depth_level" in df.columns or "depth" in df.columns:
    depth_col = "depth_level" if "depth_level" in df.columns else "depth"
    plt.figure(figsize=(8, 5))
    sns.countplot(x=depth_col, data=df, palette="coolwarm")
    plt.title("Citation Depth Distribution")
    plt.xlabel("Depth Level (0 = Base Paper)")
    plt.ylabel("Count of Papers")
    plt.tight_layout()
    plt.savefig(f"{SAVE_DIR}/depth_distribution.png", dpi=300)
    plt.close()

# ---------- 5️⃣ TF-IDF & Similarity Statistics ----------
def compute_stats(series):
    s = series.dropna()
    if s.empty:
        return pd.Series({"mean": np.nan, "mode": np.nan, "highest": np.nan, "lowest": np.nan, "median": np.nan})
    return pd.Series({
        "mean": np.mean(s),
        "mode": mode(s),
        "highest": np.max(s),
        "lowest": np.min(s),
        "median": np.median(s)
    })

# Compute for TF-IDF
if "tfidf_score" in df.columns:
    tfidf_summary = (
        df.groupby("sub_category")["tfidf_score"]
        .apply(compute_stats)
        .reset_index()
    )
    tfidf_summary.to_excel(f"{SAVE_DIR}/tfidf_stats.xlsx", index=False)

    plt.figure(figsize=(12, 6))
    sns.boxplot(x="sub_category", y="tfidf_score", data=df)
    plt.title("TF-IDF Score Distribution per Combination")
    plt.xticks(rotation=45, ha="right")
    plt.tight_layout()
    plt.savefig(f"{SAVE_DIR}/tfidf_boxplot.png", dpi=300)
    plt.close()

# Compute for Similarity (if available)
if "similarity" in df.columns:
    sim_summary = (
        df.groupby("sub_category")["similarity"]
        .apply(compute_stats)
        .reset_index()
    )
    sim_summary.to_excel(f"{SAVE_DIR}/similarity_stats.xlsx", index=False)

    plt.figure(figsize=(12, 6))
    sns.boxplot(x="sub_category", y="similarity", data=df, color="lightgreen")
    plt.title("Similarity Score Distribution per Combination")
    plt.xticks(rotation=45, ha="right")
    plt.tight_layout()
    plt.savefig(f"{SAVE_DIR}/similarity_boxplot.png", dpi=300)
    plt.close()

# ---------- Combined TF-IDF + Similarity (if both exist) ----------
if "tfidf_score" in df.columns and "similarity" in df.columns:
    df["combined_score"] = 0.6 * df["tfidf_score"].fillna(0) + 0.4 * df["similarity"].fillna(0)
    combined_summary = (
        df.groupby("sub_category")["combined_score"]
        .apply(compute_stats)
        .reset_index()
    )
    combined_summary.to_excel(f"{SAVE_DIR}/combined_stats.xlsx", index=False)

    plt.figure(figsize=(12, 6))
    sns.boxplot(x="sub_category", y="combined_score", data=df, palette="mako")
    plt.title("Combined Score (TF-IDF + Similarity) per Combination")
    plt.xticks(rotation=45, ha="right")
    plt.tight_layout()
    plt.savefig(f"{SAVE_DIR}/combined_boxplot.png", dpi=300)
    plt.close()

print(f"📊 All analyses complete. Results saved in '{SAVE_DIR}' folder.")

📘 Loading data from results_multilevel_ranked/all_multilevel_ranked_combined.xlsx ...


/var/folders/2m/_9mkxd6x3_d26ct7x886r2zw0000gn/T/ipykernel_7995/2671012309.py:58: FutureWarning: 

Passing `palette` without assigning `hue` is deprecated and will be removed in v0.14.0. Assign the `y` variable to `hue` and set `legend=False` for the same effect.

  sns.barplot(y="keyword", x="count", data=set_df, palette="Blues_r")
/opt/anaconda3/envs/SQL_tutor/lib/python3.12/site-packages/upsetplot/data.py:385: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df.fillna(False, inplace=True)
/opt/anaconda3/envs/SQL_tutor/lib/python3.12/site-packages/upsetplot/plotting.py:795: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never

📊 All analyses complete. Results saved in 'analysis_multilevel_outputs' folder.


<Figure size 1000x600 with 0 Axes>